# 🧍 Proyecto Integrador — Estimación de Pose y Despliegue en HF Spaces

**Materiales desarrollados por Matías Barreto, 2026**  
**Tecnicatura Superior en Ciencias de Datos e IA, IFTS24**  
* **Nomenclatura Oficial:** Procesamiento Digital de Imágenes  
* **Nombre de Trabajo:** Laboratorio de Tecnologías de la Imagen Digital  

---

## El proyecto

Este cuaderno es diferente a los anteriores. No es un tutorial paso a paso: es un **proyecto integrador**.

Vamos a partir de un código base funcional para detectar pose corporal con MediaPipe, y desde ahí vamos a construir y desplegar una aplicación web completa. El producto final va a estar publicado en Hugging Face Spaces y versionado en un repositorio de GitHub.

**Al completar este proyecto vamos a haber:**

1. Explorado MediaPipe Pose — la tercera solución de detección que vemos en esta unidad (además de Face Mesh y Hands).
2. Adaptado código de Jupyter a un script `app.py` listo para producción.
3. Desplegado una aplicación de visión artificial accesible desde cualquier navegador.
4. Publicado el código fuente en un repositorio de GitHub.

> ◈ Este cuaderno usa el **Cheatsheet de HF Spaces** como referencia para el despliegue. Conviene tenerlo abierto en otra pestaña: `Extras/Guias/HuggingFace-Spaces/Cheatsheet_Desarrollo_Space.ipynb`

## Microglosario

| Término | Definición | Analogía |
|---|---|---|
| **Pose estimation** | Detección automática de la posición de las articulaciones del cuerpo en una imagen | Como cuando un entrenador marca con stickers los puntos clave del cuerpo de un atleta para analizar su técnica |
| **Keypoint / Punto clave** | Coordenada que representa una articulación o punto anatómico específico (nariz, hombro, rodilla...) | Como los pines de un maniquí articulado: cada uno representa una unión móvil |
| **Visibilidad** | Valor entre 0 y 1 que indica qué tan seguro está el modelo de que ese punto es visible en la imagen | Como la confianza con la que un médico marca un punto en una radiografía: 1.0 = certeza total, 0.0 = pura suposición |
| **`app.py`** | Script Python que contiene la lógica completa de la aplicación, listo para ejecutarse fuera de Jupyter | Como el plano de una casa: en Jupyter dibujamos bocetos, en `app.py` está el plano final para construir |
| **Space (HF)** | Servidor gratuito de Hugging Face que ejecuta y publica aplicaciones Gradio | Como un hosting web especializado en aplicaciones de IA: subís el código y ellos lo sirven al mundo |

## ✦ MediaPipe Pose: los 33 puntos del cuerpo

MediaPipe Pose detecta **33 puntos clave** distribuidos por todo el cuerpo. A diferencia de Face Mesh (478 puntos en el rostro) o Hands (21 puntos en la mano), Pose cubre el esqueleto completo con menos puntos pero mayor alcance anatómico.

```
                [0] nariz
                   |
         [12] ─────┼───── [11]     ← hombros
          |                 |
         [14]              [13]    ← codos
          |                 |
         [16]              [15]    ← muñecas


         [24] ─────────── [23]     ← caderas
          |                 |
         [26]              [25]    ← rodillas
          |                 |
         [28]              [27]    ← tobillos
```

*Nota: los índices siguen la convención MediaPipe — lado derecho de la persona en índices pares, lado izquierdo en impares.*

Cada punto tiene cuatro valores:

| Atributo | Tipo | Descripción |
|---|---|---|
| `x` | float 0–1 | Posición horizontal normalizada |
| `y` | float 0–1 | Posición vertical normalizada |
| `z` | float | Profundidad relativa (aproximada) |
| `visibility` | float 0–1 | Confianza de que el punto es visible |

**Casos de uso:** análisis de postura, entrenamiento deportivo, ergonomía en el trabajo, coreografías, fisioterapia.

## Paso 1 — Instalación

Las mismas herramientas que ya conocemos. Si ya las instalaron en este entorno, la celda termina rápido.

In [9]:
!pip install gradio mediapipe opencv-python-headless numpy --quiet

import mediapipe as mp
import gradio as gr
import cv2
import numpy as np

version_mediapipe = mp.__version__
version_gradio    = gr.__version__
version_opencv    = cv2.__version__

print("✓ Entorno listo.")
print(f"  mediapipe  {version_mediapipe}")
print(f"  gradio     {version_gradio}")
print(f"  opencv     {version_opencv}")

✓ Entorno listo.
  mediapipe  0.10.35
  gradio     6.19.0
  opencv     4.13.0



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\cmpat\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


## Código base — detector de pose

La siguiente celda contiene la función central del proyecto. Algunas líneas están completas; otras tienen un `# TODO` donde ustedes van a tener que escribir.

Lean la función completa antes de ejecutarla. Los `# TODO` son parte de la consigna — no los salteen.

In [ ]:
import mediapipe as mp
import numpy as np
import cv2
import gradio as gr
import os
import math
from urllib.request import urlopen

# ── Descargar el modelo si no existe ─────────────────────────────────────

modelo_path = "pose_landmarker_heavy.task"
if not os.path.exists(modelo_path):
    print("Descargando modelo de Pose Landmarker...")
    url = "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_heavy/float16/1/pose_landmarker_heavy.task"
    try:
        with urlopen(url) as response:
            with open(modelo_path, "wb") as out_file:
                out_file.write(response.read())
        print("Modelo descargado correctamente.")
    except Exception as e:
        print(f"No se pudo descargar el modelo: {e}")
        modelo_path = None

# ── Inicialización del detector ─────────────────────────────────────────

from mediapipe.tasks.python import vision
from mediapipe.tasks.python.vision import drawing_utils, drawing_styles
from mediapipe.tasks import python

if modelo_path:
    base_options = python.BaseOptions(model_asset_path=modelo_path)
    options = vision.PoseLandmarkerOptions(
        base_options=base_options,
        output_segmentation_masks=False,
        min_pose_detection_confidence=0.5,
        min_pose_presence_confidence=0.5,
    )
    detector_pose = vision.PoseLandmarker.create_from_options(options)
    print("Detector de Pose inicializado (MediaPipe Tasks API).")
else:
    detector_pose = None
    print("No se pudo inicializar el detector de pose.")


# ── Funciones auxiliares biomecánicas ───────────────────────────────────

def dibujar_esqueleto(imagen_rgb, detection_result):
    """Dibuja landmarks y conexiones del esqueleto usando MediaPipe."""
    pose_landmarks_list = detection_result.pose_landmarks
    annotated_image = np.copy(imagen_rgb)

    pose_landmark_style = drawing_styles.get_default_pose_landmarks_style()
    pose_connection_style = drawing_utils.DrawingSpec(color=(0, 255, 0), thickness=2)

    for pose_landmarks in pose_landmarks_list:
        drawing_utils.draw_landmarks(
            image=annotated_image,
            landmark_list=pose_landmarks,
            connections=vision.PoseLandmarksConnections.POSE_LANDMARKS,
            landmark_drawing_spec=pose_landmark_style,
            connection_drawing_spec=pose_connection_style,
        )

    return annotated_image


def angulo_3p(a, b, c):
    """Ángulo ABC en grados usando producto escalar."""
    ba = a - b
    bc = c - b
    den = (np.linalg.norm(ba) * np.linalg.norm(bc)) + 1e-8
    cosang = np.clip(np.dot(ba, bc) / den, -1.0, 1.0)
    return float(np.degrees(np.arccos(cosang)))


def evaluar_postura_estatica(lista_landmarks):
    idx = {
        "hombro_i": 11,
        "hombro_d": 12,
        "codo_i": 13,
        "codo_d": 14,
        "muneca_i": 15,
        "muneca_d": 16,
        "cadera_i": 23,
        "cadera_d": 24,
        "rodilla_i": 25,
        "rodilla_d": 26,
        "tobillo_i": 27,
        "tobillo_d": 28,
    }

    # Filtro de confianza para evitar falsos positivos.
    puntos_requeridos = [
        "hombro_i", "hombro_d", "cadera_i", "cadera_d",
        "rodilla_i", "rodilla_d", "tobillo_i", "tobillo_d",
    ]
    vis_min = 0.65
    visibilidades = [lista_landmarks[idx[p]].visibility for p in puntos_requeridos]
    if any(v < vis_min for v in visibilidades):
        return {
            "ok": False,
            "diagnostico": "No evaluable por baja visibilidad en puntos clave.",
            "feedback": ["Repeti la foto con cuerpo completo y buena iluminacion."],
            "metricas": {},
        }

    def xy(nombre):
        lm = lista_landmarks[idx[nombre]]
        return np.array([lm.x, lm.y], dtype=np.float32)

    h_i, h_d = xy("hombro_i"), xy("hombro_d")
    c_i, c_d = xy("cadera_i"), xy("cadera_d")
    r_i, r_d = xy("rodilla_i"), xy("rodilla_d")
    t_i, t_d = xy("tobillo_i"), xy("tobillo_d")
    co_i, co_d = xy("codo_i"), xy("codo_d")
    m_i, m_d = xy("muneca_i"), xy("muneca_d")

    # Angulos articulares criticos.
    ang_codo_i = angulo_3p(h_i, co_i, m_i)
    ang_codo_d = angulo_3p(h_d, co_d, m_d)
    ang_rodilla_i = angulo_3p(c_i, r_i, t_i)
    ang_rodilla_d = angulo_3p(c_d, r_d, t_d)
    ang_cadera_i = angulo_3p(h_i, c_i, r_i)
    ang_cadera_d = angulo_3p(h_d, c_d, r_d)

    # Alineacion y desviacion de ejes.
    asim_hombros_y = abs(h_i[1] - h_d[1])
    asim_caderas_y = abs(c_i[1] - c_d[1])
    desvio_rod_tob_i_x = abs(r_i[0] - t_i[0])
    desvio_rod_tob_d_x = abs(r_d[0] - t_d[0])

    # Torso respecto a vertical.
    hombro_mid = (h_i + h_d) / 2.0
    cadera_mid = (c_i + c_d) / 2.0
    torso_inclinacion = angulo_3p(cadera_mid + np.array([0.0, -1.0], dtype=np.float32), cadera_mid, hombro_mid)

    # Robustez de plano para analisis bilateral.
    vis_izq = np.mean([
        lista_landmarks[idx["hombro_i"]].visibility,
        lista_landmarks[idx["cadera_i"]].visibility,
        lista_landmarks[idx["rodilla_i"]].visibility,
    ])
    vis_der = np.mean([
        lista_landmarks[idx["hombro_d"]].visibility,
        lista_landmarks[idx["cadera_d"]].visibility,
        lista_landmarks[idx["rodilla_d"]].visibility,
    ])
    alerta_plano = abs(vis_izq - vis_der) > 0.35

    tol = {
        "torso_max": 18.0,
        "asim_hombros_y_max": 0.04,
        "asim_caderas_y_max": 0.04,
        "desvio_rod_tob_x_max": 0.06,
        "rodilla_ext_min": 155.0,
        "cadera_ext_min": 150.0,
    }

    feedback = []

    if torso_inclinacion > tol["torso_max"]:
        feedback.append("Intenta enderezar la espalda.")
    if asim_hombros_y > tol["asim_hombros_y_max"]:
        feedback.append("Nivela los hombros.")
    if asim_caderas_y > tol["asim_caderas_y_max"]:
        feedback.append("Nivela la pelvis.")
    if desvio_rod_tob_i_x > tol["desvio_rod_tob_x_max"] or desvio_rod_tob_d_x > tol["desvio_rod_tob_x_max"]:
        feedback.append("Alinea rodillas con tobillos.")
    if ang_rodilla_i < tol["rodilla_ext_min"] or ang_rodilla_d < tol["rodilla_ext_min"]:
        feedback.append("Extiende un poco mas las rodillas.")
    if ang_cadera_i < tol["cadera_ext_min"] or ang_cadera_d < tol["cadera_ext_min"]:
        feedback.append("Abre la cadera para una postura mas neutra.")
    if alerta_plano:
        feedback.append("La foto no parece en plano ideal para analisis bilateral.")

    if len(feedback) == 0:
        diagnostico = "Buena postura estatica."
    elif len(feedback) <= 2:
        diagnostico = "Postura aceptable con ajustes."
    else:
        diagnostico = "Postura mejorable."

    return {
        "ok": True,
        "diagnostico": diagnostico,
        "feedback": feedback if feedback else ["Sin correcciones relevantes."],
        "metricas": {
            "torso_inclinacion": round(torso_inclinacion, 1),
            "asim_hombros_y": round(asim_hombros_y, 3),
            "asim_caderas_y": round(asim_caderas_y, 3),
            "desvio_rod_tob_i_x": round(desvio_rod_tob_i_x, 3),
            "desvio_rod_tob_d_x": round(desvio_rod_tob_d_x, 3),
            "ang_codo_i": round(ang_codo_i, 1),
            "ang_codo_d": round(ang_codo_d, 1),
            "ang_rodilla_i": round(ang_rodilla_i, 1),
            "ang_rodilla_d": round(ang_rodilla_d, 1),
            "ang_cadera_i": round(ang_cadera_i, 1),
            "ang_cadera_d": round(ang_cadera_d, 1),
        },
    }


# ── Funcion principal ────────────────────────────────────────────────────

def detectar_pose(imagen_entrada):
    """Recibe imagen RGB y devuelve imagen anotada + reporte biomecanico."""
    if detector_pose is None:
        return imagen_entrada.copy(), "Error: modelo no disponible"

    try:
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=imagen_entrada)
        detection_result = detector_pose.detect(mp_image)
        imagen_anotada = dibujar_esqueleto(imagen_entrada, detection_result)

        if not detection_result.pose_landmarks or len(detection_result.pose_landmarks) == 0:
            mensaje = "No se detecto figura humana. Asegurate de:\n"
            mensaje += "- Estar de frente a la camara\n"
            mensaje += "- Buena iluminacion\n"
            mensaje += "- Postura completamente visible"
            return imagen_anotada, mensaje

        lista_landmarks = detection_result.pose_landmarks[0]
        resultado = evaluar_postura_estatica(lista_landmarks)

        if not resultado["ok"]:
            return imagen_anotada, resultado["diagnostico"] + "\n" + "\n".join(resultado["feedback"])

        metricas = resultado["metricas"]
        feedback = "\n".join(f"- {msg}" for msg in resultado["feedback"])

        reporte = f"""
DIAGNOSTICO:
{resultado['diagnostico']}

METRICAS:
- Torso (deg): {metricas['torso_inclinacion']}
- Asimetria hombros Y: {metricas['asim_hombros_y']}
- Asimetria caderas Y: {metricas['asim_caderas_y']}
- Desvio rodilla-tobillo izq X: {metricas['desvio_rod_tob_i_x']}
- Desvio rodilla-tobillo der X: {metricas['desvio_rod_tob_d_x']}
- Angulo codo izq: {metricas['ang_codo_i']}
- Angulo codo der: {metricas['ang_codo_d']}
- Angulo rodilla izq: {metricas['ang_rodilla_i']}
- Angulo rodilla der: {metricas['ang_rodilla_d']}
- Angulo cadera izq: {metricas['ang_cadera_i']}
- Angulo cadera der: {metricas['ang_cadera_d']}

FEEDBACK:
{feedback}
"""

        return imagen_anotada, reporte.strip()

    except Exception as e:
        return imagen_entrada.copy(), f"Error al procesar: {str(e)}"


# ── Interfaz de prueba ───────────────────────────────────────────────────

interfaz_pose = gr.Interface(
    fn=detectar_pose,
    inputs=gr.Image(label="Fotografia", type="numpy"),
    outputs=[
        gr.Image(label="Pose detectada"),
        gr.Textbox(label="Informe biomecanico"),
    ],
    title="Detector de Pose - MediaPipe",
    description="Subi una imagen. El modelo detecta landmarks y evalua postura estatica.",
    flagging_mode="never",
)

interfaz_pose.launch()

✓ Detector de Pose inicializado (MediaPipe Tasks API).
* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


c:\Patricio-Velasquez-Christian-PDI-1c-2026\008\002 - PRA\008 - vision_artificial_aplicada\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Patricio-Velasquez-Christian-PDI-1c-2026\008\002 - PRA\008 - vision_artificial_aplicada\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Patricio-Velasquez-Christian-PDI-1c-2026\008\002 - PRA\008 - vision_artificial_aplicada\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


## De Jupyter a `app.py`

Un cuaderno Jupyter es un excelente entorno de exploración, pero no es lo que esperan los servidores de producción. Para desplegar en Hugging Face Spaces necesitamos un script Python clásico: `app.py`.

La lógica es la misma que ya conocemos de la unidad anterior — **arquitectura de 3 capas**:

```
┌──────────────────────────────────────────┐
│  CAPA 1 — Data Layer                     │
│  Carga única del modelo en memoria       │
│  (se ejecuta una sola vez al iniciar)    │
└──────────────────┬───────────────────────┘
                   ↓
┌──────────────────────────────────────────┐
│  CAPA 2 — Business Logic                 │
│  Función que procesa cada imagen         │
│  (se llama cada vez que llega una foto)  │
└──────────────────┬───────────────────────┘
                   ↓
┌──────────────────────────────────────────┐
│  CAPA 3 — Presentation Layer             │
│  Interfaz Gradio declarada con Blocks    │
│  (define cómo se ve la app)              │
└──────────────────────────────────────────┘
```

> ◈ Para los detalles de git y deploy, usen el Cheatsheet:
> `Extras/Guias/HuggingFace-Spaces/Cheatsheet_Desarrollo_Space.ipynb`

La celda siguiente genera los archivos de la aplicación directamente desde el cuaderno.

In [ ]:
# Esta celda genera los archivos del proyecto en una carpeta local.
# Una vez generados, esa carpeta se sube a Hugging Face Spaces con git.

import os

NOMBRE_PROYECTO = "mi-app-gym"
os.makedirs(NOMBRE_PROYECTO, exist_ok=True)
print(f"Carpeta creada: {NOMBRE_PROYECTO}/")

contenido_app = '''
# -*- coding: utf-8 -*-
import os
import numpy as np
import mediapipe as mp
import gradio as gr
from urllib.request import urlopen

from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.vision import drawing_utils, drawing_styles


def angulo_3p(a, b, c):
    ba = a - b
    bc = c - b
    den = (np.linalg.norm(ba) * np.linalg.norm(bc)) + 1e-8
    cosang = np.clip(np.dot(ba, bc) / den, -1.0, 1.0)
    return float(np.degrees(np.arccos(cosang)))


def evaluar_postura_estatica(lista_landmarks):
    idx = {
        "hombro_i": 11, "hombro_d": 12,
        "codo_i": 13, "codo_d": 14,
        "muneca_i": 15, "muneca_d": 16,
        "cadera_i": 23, "cadera_d": 24,
        "rodilla_i": 25, "rodilla_d": 26,
        "tobillo_i": 27, "tobillo_d": 28,
    }

    puntos_requeridos = ["hombro_i", "hombro_d", "cadera_i", "cadera_d", "rodilla_i", "rodilla_d", "tobillo_i", "tobillo_d"]
    vis_min = 0.65
    visibilidades = [lista_landmarks[idx[p]].visibility for p in puntos_requeridos]
    if any(v < vis_min for v in visibilidades):
        return {
            "ok": False,
            "diagnostico": "No evaluable por baja visibilidad en puntos clave.",
            "feedback": ["Repeti la foto con cuerpo completo y buena iluminacion."],
            "metricas": {},
        }

    def xy(nombre):
        lm = lista_landmarks[idx[nombre]]
        return np.array([lm.x, lm.y], dtype=np.float32)

    h_i, h_d = xy("hombro_i"), xy("hombro_d")
    c_i, c_d = xy("cadera_i"), xy("cadera_d")
    r_i, r_d = xy("rodilla_i"), xy("rodilla_d")
    t_i, t_d = xy("tobillo_i"), xy("tobillo_d")
    co_i, co_d = xy("codo_i"), xy("codo_d")
    m_i, m_d = xy("muneca_i"), xy("muneca_d")

    ang_codo_i = angulo_3p(h_i, co_i, m_i)
    ang_codo_d = angulo_3p(h_d, co_d, m_d)
    ang_rodilla_i = angulo_3p(c_i, r_i, t_i)
    ang_rodilla_d = angulo_3p(c_d, r_d, t_d)
    ang_cadera_i = angulo_3p(h_i, c_i, r_i)
    ang_cadera_d = angulo_3p(h_d, c_d, r_d)

    asim_hombros_y = abs(h_i[1] - h_d[1])
    asim_caderas_y = abs(c_i[1] - c_d[1])
    desvio_rod_tob_i_x = abs(r_i[0] - t_i[0])
    desvio_rod_tob_d_x = abs(r_d[0] - t_d[0])

    hombro_mid = (h_i + h_d) / 2.0
    cadera_mid = (c_i + c_d) / 2.0
    torso_inclinacion = angulo_3p(cadera_mid + np.array([0.0, -1.0], dtype=np.float32), cadera_mid, hombro_mid)

    vis_izq = np.mean([lista_landmarks[idx["hombro_i"]].visibility, lista_landmarks[idx["cadera_i"]].visibility, lista_landmarks[idx["rodilla_i"]].visibility])
    vis_der = np.mean([lista_landmarks[idx["hombro_d"]].visibility, lista_landmarks[idx["cadera_d"]].visibility, lista_landmarks[idx["rodilla_d"]].visibility])
    alerta_plano = abs(vis_izq - vis_der) > 0.35

    tol = {
        "torso_max": 18.0,
        "asim_hombros_y_max": 0.04,
        "asim_caderas_y_max": 0.04,
        "desvio_rod_tob_x_max": 0.06,
        "rodilla_ext_min": 155.0,
        "cadera_ext_min": 150.0,
    }

    feedback = []
    if torso_inclinacion > tol["torso_max"]:
        feedback.append("Intenta enderezar la espalda.")
    if asim_hombros_y > tol["asim_hombros_y_max"]:
        feedback.append("Nivela los hombros.")
    if asim_caderas_y > tol["asim_caderas_y_max"]:
        feedback.append("Nivela la pelvis.")
    if desvio_rod_tob_i_x > tol["desvio_rod_tob_x_max"] or desvio_rod_tob_d_x > tol["desvio_rod_tob_x_max"]:
        feedback.append("Alinea rodillas con tobillos.")
    if ang_rodilla_i < tol["rodilla_ext_min"] or ang_rodilla_d < tol["rodilla_ext_min"]:
        feedback.append("Extiende un poco mas las rodillas.")
    if ang_cadera_i < tol["cadera_ext_min"] or ang_cadera_d < tol["cadera_ext_min"]:
        feedback.append("Abre la cadera para una postura mas neutra.")
    if alerta_plano:
        feedback.append("La foto no parece en plano ideal para analisis bilateral.")

    if len(feedback) == 0:
        diagnostico = "Buena postura estatica."
    elif len(feedback) <= 2:
        diagnostico = "Postura aceptable con ajustes."
    else:
        diagnostico = "Postura mejorable."

    return {
        "ok": True,
        "diagnostico": diagnostico,
        "feedback": feedback if feedback else ["Sin correcciones relevantes."],
        "metricas": {
            "torso_inclinacion": round(torso_inclinacion, 1),
            "asim_hombros_y": round(asim_hombros_y, 3),
            "asim_caderas_y": round(asim_caderas_y, 3),
            "desvio_rod_tob_i_x": round(desvio_rod_tob_i_x, 3),
            "desvio_rod_tob_d_x": round(desvio_rod_tob_d_x, 3),
            "ang_codo_i": round(ang_codo_i, 1),
            "ang_codo_d": round(ang_codo_d, 1),
            "ang_rodilla_i": round(ang_rodilla_i, 1),
            "ang_rodilla_d": round(ang_rodilla_d, 1),
            "ang_cadera_i": round(ang_cadera_i, 1),
            "ang_cadera_d": round(ang_cadera_d, 1),
        },
    }


def dibujar_esqueleto(imagen_rgb, detection_result):
    pose_landmarks_list = detection_result.pose_landmarks
    annotated_image = np.copy(imagen_rgb)
    pose_landmark_style = drawing_styles.get_default_pose_landmarks_style()
    pose_connection_style = drawing_utils.DrawingSpec(color=(0, 255, 0), thickness=2)

    for pose_landmarks in pose_landmarks_list:
        drawing_utils.draw_landmarks(
            image=annotated_image,
            landmark_list=pose_landmarks,
            connections=vision.PoseLandmarksConnections.POSE_LANDMARKS,
            landmark_drawing_spec=pose_landmark_style,
            connection_drawing_spec=pose_connection_style,
        )

    return annotated_image


modelo_path = "pose_landmarker_heavy.task"
if not os.path.exists(modelo_path):
    url = "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_heavy/float16/1/pose_landmarker_heavy.task"
    with urlopen(url) as response:
        with open(modelo_path, "wb") as out_file:
            out_file.write(response.read())

base_options = python.BaseOptions(model_asset_path=modelo_path)
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    output_segmentation_masks=False,
    min_pose_detection_confidence=0.9,
    min_pose_presence_confidence=0.9,
)
detector_pose = vision.PoseLandmarker.create_from_options(options)


def analizar_postura_fitness(imagen_entrada):
    try:
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=imagen_entrada)
        detection_result = detector_pose.detect(mp_image)
        imagen_anotada = dibujar_esqueleto(imagen_entrada, detection_result)

        if not detection_result.pose_landmarks or len(detection_result.pose_landmarks) == 0:
            return imagen_anotada, "No se detecto figura humana."

        resultado = evaluar_postura_estatica(detection_result.pose_landmarks[0])
        if not resultado["ok"]:
            return imagen_anotada, resultado["diagnostico"] + "\n" + "\n".join(resultado["feedback"])

        m = resultado["metricas"]
        f = "\n".join(f"- {msg}" for msg in resultado["feedback"])

        reporte = f"""
DIAGNOSTICO:
{resultado['diagnostico']}

METRICAS:
- Torso (deg): {m['torso_inclinacion']}
- Asimetria hombros Y: {m['asim_hombros_y']}
- Asimetria caderas Y: {m['asim_caderas_y']}
- Desvio rodilla-tobillo izq X: {m['desvio_rod_tob_i_x']}
- Desvio rodilla-tobillo der X: {m['desvio_rod_tob_d_x']}
- Angulo codo izq: {m['ang_codo_i']}
- Angulo codo der: {m['ang_codo_d']}
- Angulo rodilla izq: {m['ang_rodilla_i']}
- Angulo rodilla der: {m['ang_rodilla_d']}
- Angulo cadera izq: {m['ang_cadera_i']}
- Angulo cadera der: {m['ang_cadera_d']}

FEEDBACK:
{f}
"""

        return imagen_anotada, reporte.strip()

    except Exception as e:
        return imagen_entrada.copy(), f"Error al procesar: {str(e)}"


with gr.Blocks(title="Analizador de Postura Fitness") as app_fitness:
    gr.Markdown("# Analizador de Postura para Entrenamiento")
    with gr.Row():
        with gr.Column():
            entrada_imagen = gr.Image(label="Foto de postura", type="numpy", sources=["upload", "webcam"])
            boton_analizar = gr.Button("Analizar postura", variant="primary")
        with gr.Column():
            salida_imagen = gr.Image(label="Esqueleto detectado")
            salida_reporte = gr.Textbox(label="Analisis detallado", lines=14, interactive=False)

    boton_analizar.click(fn=analizar_postura_fitness, inputs=entrada_imagen, outputs=[salida_imagen, salida_reporte])


if __name__ == "__main__":
    app_fitness.launch(server_name="0.0.0.0", server_port=7860, share=False)
'''

ruta_app = os.path.join(NOMBRE_PROYECTO, "app.py")
with open(ruta_app, "w", encoding="utf-8") as archivo_app:
    archivo_app.write(contenido_app)

print("app.py generado con logica biomecanica estatica mejorada.")

✓ Carpeta creada: mi-app-gym/
✓ app.py generado (usando instructivo oficial de MediaPipe).
  Completá los TODO antes de hacer el deploy.


In [17]:
# Generamos el requirements.txt — lista de dependencias del proyecto
# Hugging Face Spaces lee este archivo para instalar lo que necesita

import os

# Definimos cada dependencia con su versión mínima
dependencias = [
    "gradio>=4.0.0",
    "mediapipe>=0.10.35",
    "opencv-python-headless>=4.8.0",
    "numpy>=1.24.0",
]

# Unimos todas las líneas con salto de línea
contenido_requirements = "\n".join(dependencias)

ruta_requirements = os.path.join(NOMBRE_PROYECTO, 'requirements.txt')
with open(ruta_requirements, 'w', encoding='utf-8') as archivo_req:
    archivo_req.write(contenido_requirements)

# Confirmamos el contenido generado
print("✓ requirements.txt generado:")
print()
for dependencia in dependencias:
    print(f"  {dependencia}")

# Mostramos los archivos que están listos para el deploy
print()
print("Archivos del proyecto:")
archivos_generados = os.listdir(NOMBRE_PROYECTO)
for nombre_archivo in archivos_generados:
    print(f"  {NOMBRE_PROYECTO}/{nombre_archivo}")

✓ requirements.txt generado:

  gradio>=4.0.0
  mediapipe>=0.10.35
  opencv-python-headless>=4.8.0
  numpy>=1.24.0

Archivos del proyecto:
  mi-app-gym/app.py
  mi-app-gym/requirements.txt


## ✎ Consigna 2 — La interfaz

El `app.py` que generamos tiene varios `# TODO` pendientes. La consigna es completarlos hasta tener una aplicación que corra sin errores.

**Pasos:**

1. Abrí `app.py` en VS Code (o cualquier editor).

2. **Completá la función `detectar_pose`** pegando la versión final que construiste en la Consigna 1 — con las métricas propias incluidas.

3. **Completá los componentes de la interfaz** (`entrada_imagen`, `salida_imagen`, `salida_texto`). Usen el Cheatsheet de Extras como referencia para ver los componentes disponibles.

4. **Probá la app localmente** desde la terminal:
   ```bash
   cd mi-pose-app
   python app.py
   ```
   Si abre el navegador y funciona, están listos para el deploy.

> ◈ **¿La función no devuelve lo que esperan?** Revisá que los componentes en `outputs=` coincidan exactamente con los valores que devuelve `detectar_pose` (imagen + texto, en ese orden).

## ✎ Consigna 3 — El despliegue

Con la app funcionando localmente, es momento de publicarla. El proceso completo está detallado en el Cheatsheet de Extras — acá va el resumen:

### En Hugging Face Spaces

1. Entrá a [huggingface.co/new-space](https://huggingface.co/new-space)
2. Elegí un nombre para el Space (puede coincidir con `NOMBRE_PROYECTO`)
3. Seleccioná **SDK: Gradio** y **Hardware: CPU free**
4. Seguí los comandos de git del Cheatsheet para vincular y subir los archivos:
   ```bash
   git init
   git add .
   git commit -m 'feat: detector de pose con MediaPipe'
   git remote add origin https://huggingface.co/spaces/TU_USUARIO/TU_SPACE
   git branch -M main
   git push -u origin main
   ```

### En GitHub

5. Creá un repositorio nuevo en [github.com/new](https://github.com/new)
6. Vinculá el mismo proyecto con un segundo remote:
   ```bash
   git remote add github https://github.com/TU_USUARIO/TU_REPO
   git push github main
   ```

> ◈ El Space en HF va a quedar público y accesible por URL. Compartí el link cuando esté desplegado.

## ✎ Para pensar

Una vez que la aplicación esté desplegada, respondé estas preguntas:

1. **Sobre el modelo:** MediaPipe Pose fue entrenado con millones de imágenes. Sin embargo, en algunas fotos falla o detecta puntos en posiciones incorrectas. ¿En qué tipo de imágenes notaste más errores? ¿A qué factores creés que se debe?

2. **Sobre la arquitectura:** El `app.py` separa la carga del modelo (Capa 1) de la función de procesamiento (Capa 2). ¿Qué pasaría si cargáramos el modelo *dentro* de `detectar_pose`, en lugar de hacerlo una sola vez al inicio? ¿Por qué eso sería un problema en producción?

3. **Sobre el deploy:** Comparando el flujo que siguieron hoy (Jupyter → `app.py` → HF Spaces + GitHub) con cómo venían trabajando, ¿qué ventajas concretas tiene este proceso? ¿Qué parte les resultó más difícil de entender o ejecutar?